In [1]:
import pandas as pd
import numpy as np
from src.utils import _norm_text, _pick_org
from src.state_of_art_models.tripartite_lightGCN import TripartiteLightGCNRunner

In [2]:
aud = pd.read_csv(
    "data/audiencies.csv",
    usecols=["audiencia_id", "sujeto_pasivo_id", "institucion_id", "materias_tratadas", "fecha"]
)
act = pd.read_csv(
    "data/active_subjects.csv",
    usecols=["audiencia_id", "Nombre completo", "Representa a", "Calidad"]
)
pas = pd.read_csv(
    "data/passive_subjects.csv",
    usecols=["id", "nombre", "cargo", "institution_id"]
).rename(columns={"id": "sujeto_pasivo_id"})
materias = pd.read_pickle("data/materias_embeddings.pkl")

# Format data

In [15]:
import pandas as pd
import numpy as np

def build_lightgcn_inputs(active, aud, passive):
    active = active.copy()
    aud = aud.copy()
    passive = passive.copy()

    # ---- normalización de usuarios (sujetos activos) ----
    active["display_name"] = active["Nombre completo"].fillna("").astype(str).str.strip()
    active["display_org"]  = active.apply(_pick_org, axis=1).astype(str).str.strip()
    active["__name_norm"]  = active["display_name"].map(_norm_text)
    active["__org_norm"]   = active["display_org"].map(_norm_text)
    active["user_key"]     = (active["__name_norm"] + " | " + active["__org_norm"]).str.strip(" |")

    # ---- timestamps de audiencias ----
    aud["timestamp"] = pd.to_datetime(aud["fecha"], errors="coerce")

    aud_cols = ["audiencia_id", "sujeto_pasivo_id", "institucion_id", "timestamp"]
    edges = active.merge(aud[aud_cols], on="audiencia_id", how="inner", suffixes=("_act", "_aud"))

    sp_col   = "sujeto_pasivo_id"
    inst_col = "institucion_id"

    edges["item_key"]       = edges[sp_col].astype(str).str.strip()
    edges["institution_id"] = edges[inst_col].astype(str).str.strip()
    edges["timestamp"]      = pd.to_datetime(edges["timestamp"], errors="coerce")
    edges = edges.dropna(subset=["user_key", "item_key", "timestamp"])

    # 1 interacción por (usuario, autoridad, día)
    edges["__date"] = edges["timestamp"].dt.date
    edges = edges.drop_duplicates(subset=["user_key", "item_key", "__date"]).drop(columns="__date")

    # índices numéricos
    user_index = (
        pd.DataFrame({"user_key": sorted(edges["user_key"].unique())})
        .reset_index()
        .rename(columns={"index": "user_id"})
    )
    item_index = (
        pd.DataFrame({"item_key": sorted(edges["item_key"].unique())})
        .reset_index()
        .rename(columns={"index": "item_id"})
    )

    edges = (
        edges.merge(user_index, on="user_key", how="left")
             .merge(item_index, on="item_key", how="left")
    )

    users_df = (
        user_index.merge(
            active[["user_key", "display_name", "display_org"]].drop_duplicates("user_key"),
            on="user_key",
            how="left",
        )
    )

    # ---- autoridades (sujetos pasivos) ----
    passive = passive.copy()
    passive["item_key"] = passive["sujeto_pasivo_id"].astype(str).str.strip()

    items_df = (
        item_index.merge(
            passive[["item_key", "nombre", "cargo", "institution_id"]],
            on="item_key",
            how="left",
        )
    )

    interactions_df = (
        edges[["audiencia_id", "user_id", "item_id", "timestamp"]]
        .sort_values(["user_id", "timestamp"])
        .reset_index(drop=True)
    )

    return interactions_df, users_df, items_df



In [16]:
interactions_df, users_df, items_df = build_lightgcn_inputs(act, aud, pas)

/var/folders/pc/1tbslm8954q5q5cyk947n34h0000gn/T/ipykernel_32477/2883879802.py:17: FutureWarning: In a future version of pandas, parsing datetimes with mixed time zones will raise an error unless `utc=True`. Please specify `utc=True` to opt in to the new behaviour and silence this warning. To create a `Series` with mixed offsets and `object` dtype, please use `apply` and `datetime.datetime.strptime`
  aud["timestamp"] = pd.to_datetime(aud["fecha"], errors="coerce")


In [5]:
from sklearn.decomposition import PCA
X = np.stack(materias["embedding"].values)
pca = PCA()             # no fijes n_components aquí
pca.fit(X)              # X = matriz original de embeddings (632k × 384)
import numpy as np

var_acum = np.cumsum(pca.explained_variance_ratio_)
n_comp_90 = np.argmax(var_acum >= 0.90) + 1
print(n_comp_90)


212


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/decomposition/_pca.py:604: RuntimeWarning: divide by zero encountered in matmul
  C = X.T @ X
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/decomposition/_pca.py:604: RuntimeWarning: overflow encountered in matmul
  C = X.T @ X
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/decomposition/_pca.py:604: RuntimeWarning: invalid value encountered in matmul
  C = X.T @ X


In [6]:
from sklearn.decomposition import PCA

X = np.stack(materias["embedding"].values)
pca = PCA(n_components=212, random_state=42)
X_pca = pca.fit_transform(X)

materias_pca = materias.copy()
materias_pca["embedding"] = list(X_pca)  # reemplazamos por PCA


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/decomposition/_pca.py:604: RuntimeWarning: divide by zero encountered in matmul
  C = X.T @ X
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/decomposition/_pca.py:604: RuntimeWarning: overflow encountered in matmul
  C = X.T @ X
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/decomposition/_pca.py:604: RuntimeWarning: invalid value encountered in matmul
  C = X.T @ X
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/decomposition/_base.py:148: RuntimeWarning: divide by zero encountered in matmul
  X_transformed = X @ self.components_.T
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/decomposition/_base.py:148: RuntimeWarning: overflow encountered in matmul
  X_transformed = X @ self.components_.T
/Library/Frameworks/Python.framework/Versions/3

In [9]:
import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)


In [11]:
from sklearn.cluster import MiniBatchKMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score

subset_idx = np.random.choice(X_pca.shape[0], size=50000, replace=False)
X_sub = X_pca[subset_idx]

for k in [100, 200, 300, 500, 600, 700, 800]:
    km = MiniBatchKMeans(n_clusters=k, random_state=42, batch_size=4096)
    lab_sub = km.fit_predict(X_sub)
    sil = silhouette_score(X_sub, lab_sub)
    db  = davies_bouldin_score(X_sub, lab_sub)
    print(f"k={k} | silhouette={sil:.4f} | DB={db:.4f}")

k=100 | silhouette=0.0514 | DB=4.0489
k=200 | silhouette=0.0528 | DB=3.8598
k=300 | silhouette=0.0524 | DB=3.7393
k=500 | silhouette=0.0583 | DB=3.4779
k=600 | silhouette=0.0506 | DB=3.3268
k=700 | silhouette=0.0550 | DB=3.1792
k=800 | silhouette=0.0401 | DB=3.0509


In [12]:
from sklearn.cluster import MiniBatchKMeans

k = 500
kmeans = MiniBatchKMeans(
    n_clusters=k,
    random_state=42,
    batch_size=4096
)

labels = kmeans.fit_predict(X_pca[:, :100].astype("float32"))
materias_pca["topic_id"] = labels


# Model training

In [17]:
# base para LightGCN: solo lo que necesita el runner
interactions_df_light = (
    interactions_df[["user_id", "item_id", "timestamp"]]
    .dropna()
    .drop_duplicates()
)

# item–topic: juntar materias (topic_id) con la autoridad (item_id) vía audiencia_id
item_topics_df = (
    materias_pca[["audiencia_id", "topic_id"]]
    .merge(
        interactions_df[["audiencia_id", "item_id"]].drop_duplicates(),
        on="audiencia_id",
        how="inner",
    )
    [["item_id", "topic_id"]]
    .drop_duplicates()
)


In [18]:
# usa las mismas dims que usaste para KMeans (puede ser todo X_pca)
X_topic = X_pca.astype("float32")   # por ejemplo, shape (N, 212)
D = X_topic.shape[1]                # aquí queda D = 212 (o lo que sea)

topic_ids = np.unique(labels)

topic_embs = []
for t in topic_ids:
    idx = np.where(labels == t)[0]
    centroid = X_topic[idx].mean(axis=0)  # shape (D,)
    topic_embs.append(centroid)

topics_df = pd.DataFrame({
    "topic_id": topic_ids,
    "embedding": topic_embs,
}).sort_values("topic_id").reset_index(drop=True)

print("Dim de los embeddings de tema:", len(topics_df["embedding"].iloc[0]))


Dim de los embeddings de tema: 212


In [ ]:
runner = TripartiteLightGCNRunner(
    dim=212,       
    n_layers=3,
    lr=1e-3,
    l2=1e-4,
    batch_size=4096,
    epochs=100,
    patience=20,
    device="cpu",
)

test_metrics = runner.fit(
    interactions_df=interactions_df,
    item_topics_df=item_topics_df,
    topics_df=topics_df,
)
print(test_metrics)

Epoch 001 | loss 0.6893 | val nDCG@10 0.0297
Epoch 002 | loss 0.6590 | val nDCG@10 0.0295
Epoch 003 | loss 0.5976 | val nDCG@10 0.0292
Epoch 004 | loss 0.5195 | val nDCG@10 0.0287
Epoch 005 | loss 0.4388 | val nDCG@10 0.0285
Epoch 006 | loss 0.3671 | val nDCG@10 0.0284
Epoch 007 | loss 0.3087 | val nDCG@10 0.0284
Epoch 008 | loss 0.2635 | val nDCG@10 0.0285
Epoch 009 | loss 0.2272 | val nDCG@10 0.0284
Epoch 010 | loss 0.2005 | val nDCG@10 0.0285
Epoch 011 | loss 0.1790 | val nDCG@10 0.0286
Epoch 012 | loss 0.1612 | val nDCG@10 0.0287
Epoch 013 | loss 0.1470 | val nDCG@10 0.0288
Epoch 014 | loss 0.1360 | val nDCG@10 0.0291
Epoch 015 | loss 0.1249 | val nDCG@10 0.0292
Epoch 016 | loss 0.1164 | val nDCG@10 0.0294
Epoch 017 | loss 0.1076 | val nDCG@10 0.0296
Epoch 018 | loss 0.1019 | val nDCG@10 0.0298
Epoch 019 | loss 0.0966 | val nDCG@10 0.0300
Epoch 020 | loss 0.0911 | val nDCG@10 0.0303
Epoch 021 | loss 0.0860 | val nDCG@10 0.0305
Epoch 022 | loss 0.0817 | val nDCG@10 0.0308
Epoch 023 